# MOSTA Virtual Video

This notebook runs the full MOSTA baseline trajectory video workflow inside the current downstream repo:

- load MOSTA data and pretrained dynamics model
- sample the observed `t=0` pool
- run split-SDE simulation from `0.0 -> 3.0`
- classify simulated cells over time
- optionally compute focus communication overlays
- render frames and export a GIF

Note: execution requires the same dynamical-model environment used by the original MOSTA pipeline. If model dependencies are missing, the import/config cells still work but the run cell will fail until that environment is available.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "assets").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate cytobridge-downstream repository root.")


DOWNSTREAM_ROOT = find_repo_root()
if str(DOWNSTREAM_ROOT) not in sys.path:
    sys.path.insert(0, str(DOWNSTREAM_ROOT))

from downstream_helpers import (
    MostaBaselineVideoConfig,
    list_output_files,
    resolve_mosta_baseline_video_output_dir,
    run_mosta_baseline_video,
)


In [ ]:
config = MostaBaselineVideoConfig(
    output_name="mosta_brain_comm_baseline_notebook",
    time_start=0.0,
    time_end=3.0,
    time_step=0.05,
    sde_n_samples=50000,
    split_sde_dt=0.05,
    split_sigma=0.03,
    split_growth_alpha=1.0,
    spatial_warp_to_observed=True,
    spatial_warp_k=8,
    spatial_warp_eps=1e-6,
    interaction_m=1024,
    classifier_epochs=500,
    classifier_hidden=128,
    classifier_n_pcs=12,
    classifier_feature_start=1,
    classifier_best_metric="bacc",
    classifier_train_on_full_data=True,
    classifier_cache=True,
    classifier_cache_path="assets/mosta/classifier_cache/classifier_resmlp_52fb7dc647bfe334.pt",
    classifier_force_load=True,
    video_style="fixed_2d",
    video_point_subsample=50000,
    gif_fps=4,
    frame_dpi=300,
    point_size=2.5,
    point_alpha=0.9,
    show_titles=False,
    draw_focus_interactions=True,
    focus_celltype="Brain",
    focus_mode="both",
    focus_top_k=3,
    focus_min_weight=0.0,
    focus_edge_width_min=1.0,
    focus_edge_width_max=2.6,
    focus_edge_alpha_min=0.25,
    focus_edge_alpha_max=0.85,
    focus_edge_curve=0.18,
    focus_edge_curve_bi=0.35,
    focus_neighbor_pct=0.2,
    focus_other_desaturate=0.0,
    focus_show_endpoints=True,
    focus_endpoint_size=26.0,
    focus_endpoint_alpha=0.95,
    focus_endpoint_edgecolor="#ffffff",
    focus_endpoint_linewidth=0.8,
    attention_stride=1,
    attention_max_cells=30000,
    attention_use_real_observed=False,
    axis_limit_mode="per_timepoint",
    axis_pad_frac=0.03,
    panel_size=4.2,
    camera_elev=28.0,
    camera_azim=-54.0,
    random_seed=42,
    clean_output=True,
)

output_dir = resolve_mosta_baseline_video_output_dir(config)
output_dir


In [ ]:
result = run_mosta_baseline_video(config)
result


In [ ]:
list_output_files(result.output_dir)


In [ ]:
summary = json.loads(result.summary_path.read_text(encoding="utf-8"))
summary


In [ ]:
display(Image(filename=str(result.gif_path)))
print(result.gif_path)


In [ ]:
pd.read_csv(result.baseline_counts_csv).head()


In [ ]:
if result.communication_sampling_csv is not None:
    pd.read_csv(result.communication_sampling_csv).head()
else:
    print("Focus communication overlay disabled.")


## Notes

- This notebook runs a complete `run -> frame -> GIF -> summary` flow rather than reconstructing from an existing run directory.
- The API entry point is `run_mosta_baseline_video(config)` from `downstream_helpers/mosta_baseline_video.py`.
- A thin CLI wrapper is also available at `scripts/mosta_baseline_video_local.py`.
